In [ ]:
import requests
import pandas as pd
import os
import time


In [ ]:


APP_TOKEN    = "YOUR TOKEN HERE"# replace with your tokenBASE_URL     = "https://data.cityofchicago.org/resource/ijzp-q8t2.json"
BATCH_SIZE   = 50000
OUTPUT_DIR   = "chicago_crime_data"
BATCH_DIR    = os.path.join(OUTPUT_DIR, "batches")
FINAL_OUTPUT = os.path.join(OUTPUT_DIR, "chicago_crimes_2001_2025.csv")

os.makedirs(BATCH_DIR, exist_ok=True)

EXPECTED_COLUMNS = [
    'id', 'case_number', 'date', 'block', 'iucr', 'primary_type',
    'description', 'location_description', 'arrest', 'domestic', 'beat',
    'district', 'ward', 'community_area', 'fbi_code', 'year', 'updated_on',
    'x_coordinate', 'y_coordinate', 'latitude', 'longitude', 'location'
]

def fetch_batch(offset, retries=3):
    params = {
        "$limit":      BATCH_SIZE,
        "$offset":     offset,
        "$order":      "date ASC",
        "$$app_token": APP_TOKEN
    }
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(BASE_URL, params=params, timeout=60)
            response.raise_for_status()
            return response.json()
        except requests.RequestException as e:
            print(f"  [Attempt {attempt}/{retries}] Request failed: {e}")
            if attempt < retries:
                time.sleep(5 * attempt)
            else:
                raise

def download_all():
    offset    = 0
    batch_num = 1
    total     = 0

    print("Starting full download...\n")

    while True:
        batch_file = os.path.join(BATCH_DIR, f"batch_{batch_num:04d}.csv")

        if os.path.exists(batch_file):
            print(f"Batch {batch_num} already exists, skipping...")
            offset    += BATCH_SIZE
            batch_num += 1
            continue

        print(f"Fetching batch {batch_num} (offset={offset})...")

        batch = fetch_batch(offset)

        if not batch:
            print("No more data. Download complete.")
            break

        batch_df = pd.DataFrame(batch)

        for col in EXPECTED_COLUMNS:
            if col not in batch_df.columns:
                batch_df[col] = None
        batch_df = batch_df[EXPECTED_COLUMNS]

        batch_df.to_csv(batch_file, index=False)

        total     += len(batch)
        offset    += BATCH_SIZE
        batch_num += 1

        print(f"  -> {len(batch)} records fetched. Total so far: {total}")
        time.sleep(0.5)

    print(f"\nDone! Total records: {total}")

def combine_batches():
    print("Combining batches into final CSV...")
    batch_files = sorted([
        os.path.join(BATCH_DIR, f) 
        for f in os.listdir(BATCH_DIR) 
        if f.endswith('.csv')
    ])

    dfs = []
    for f in batch_files:
        print(f"Loading {f}...")
        dfs.append(pd.read_csv(f, low_memory=False))

    df_final = pd.concat(dfs, ignore_index=True)
    df_final.to_csv(FINAL_OUTPUT, index=False)
    print(f"Saved {len(df_final)} rows to {FINAL_OUTPUT}")

download_all()
combine_batches()